# Masked Diffusion Language Model — Training on Colab Pro

**Paper:** [Discrete Diffusion Language Model for Efficient Text Summarization](https://arxiv.org/abs/2407.10998) (NAACL 2025)

**Runtime:** GPU (A100 / V100 / T4)

---

**Checkpoints save to Google Drive** — training survives session restarts.

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive (checkpoints will be saved here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository
!git clone -b feature/architecture_update https://github.com/KrugD/Final_qualification_work.git
%cd Final_qualification_work
!ls src/

In [ ]:
# Install core dependencies
!pip install accelerate transformers datasets pyyaml python-dotenv \
    tqdm numpy sentencepiece protobuf rouge-score bert-score \
    comet_ml pandas --quiet

In [ ]:
# Install mamba-ssm with proper build
# Step 1: Install build tools
!pip install ninja cmake packaging --quiet

# Step 2: Install causal-conv1d first
!pip install causal-conv1d --no-build-isolation --quiet

# Step 3: Install mamba-ssm
!pip install mamba-ssm --no-build-isolation --quiet

# This may take 10-20 min (compiling CUDA kernels)

In [ ]:
# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Check mamba
try:
    from mamba_ssm import Mamba
    print("\nmamba-ssm: INSTALLED ✓")
except ImportError as e:
    print(f"\nmamba-ssm: NOT available ({e})")
    print("Training will use FallbackMamba (still works correctly)")

# Check project
from src.model import MaskedDiffusionSummarizer
print("Project modules: OK ✓")

In [ ]:
# Auto-detect GPU and set optimal batch size
import torch
import yaml

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if vram_gb >= 35:  # A100 40GB
    batch_size, grad_accum = 16, 2  # effective = 32
elif vram_gb >= 14:  # V100 16GB / T4 16GB
    batch_size, grad_accum = 8, 4   # effective = 32
else:  # Smaller GPU
    batch_size, grad_accum = 4, 8   # effective = 32

print(f"GPU: {gpu_name} ({vram_gb:.0f} GB)")
print(f"Batch size: {batch_size}, Grad accumulation: {grad_accum}")
print(f"Effective batch: {batch_size * grad_accum}")

# Update config
with open('config/colab_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['training']['batch_size'] = batch_size
config['training']['gradient_accumulation_steps'] = grad_accum

# Use bf16 for A100 (better than fp16)
if 'A100' in gpu_name:
    config['training']['mixed_precision'] = 'bf16'
    print("Using bf16 precision (A100)")

with open('config/colab_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\nConfig updated!")

In [ ]:
# Setup CometML
import os

os.environ["COMET_API_KEY"] = "q8ISQpceIfRi7PqwBptiITAba"
os.environ["COMET_PROJECT_NAME"] = "diffusion-summarization"

with open(".env", "w") as f:
    f.write(f'COMET_API_KEY={os.environ["COMET_API_KEY"]}\n')
    f.write(f'COMET_PROJECT_NAME={os.environ["COMET_PROJECT_NAME"]}\n')

print("CometML configured ✓")
print("Dashboard: https://www.comet.com")

## 2. Train

In [ ]:
# Start training (checkpoints save to Google Drive)
!python train.py --config config/colab_config.yaml

## 2b. Resume Training (after session restart)

If session disconnected — re-run cells 1-6 (setup), then resume below.

In [ ]:
# Find latest checkpoint on Google Drive
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive/diffusion_checkpoints")
if checkpoint_dir.exists():
    # List all checkpoints
    checkpoints = sorted(checkpoint_dir.glob("checkpoint_*"))
    best_model = checkpoint_dir / "best_model"
    
    print("Available checkpoints:")
    for cp in checkpoints:
        print(f"  {cp.name}")
    if best_model.exists():
        import torch
        metrics_path = best_model / "best_metrics.pt"
        if metrics_path.exists():
            m = torch.load(metrics_path, weights_only=False)
            print(f"\n  best_model (eval_loss={m.get('eval_loss', '?'):.4f}, epoch={m.get('epoch', '?')})")
    
    if checkpoints:
        latest = checkpoints[-1]
        print(f"\nLatest: {latest.name}")
        print(f"\nRun this to resume:")
        print(f'!python train.py --config config/colab_config.yaml --resume "{latest}"')
else:
    print("No checkpoints found. Start training from scratch (cell above).")

In [ ]:
# Resume training from latest checkpoint
# Uncomment and update path:
# !python train.py --config config/colab_config.yaml --resume "/content/drive/MyDrive/diffusion_checkpoints/checkpoint_epoch2_step4000"

## 3. Check Results

In [ ]:
import torch
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive/diffusion_checkpoints")

# Best model metrics
best_metrics_path = checkpoint_dir / "best_model" / "best_metrics.pt"
if best_metrics_path.exists():
    metrics = torch.load(best_metrics_path, weights_only=False)
    print("=" * 50)
    print("BEST MODEL METRICS")
    print("=" * 50)
    for k, v in sorted(metrics.items()):
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print("Best model not saved yet.")

# Training summary
summary_path = checkpoint_dir / "training_summary.pt"
if summary_path.exists():
    summary = torch.load(summary_path, weights_only=False)
    print("\n" + "=" * 50)
    print("TRAINING SUMMARY")
    print("=" * 50)
    for k, v in summary.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 4. Test Generation

In [ ]:
import torch
from src.model import MaskedDiffusionSummarizer
from transformers import AutoTokenizer

device = torch.device("cuda")
model = MaskedDiffusionSummarizer.from_pretrained(
    "/content/drive/MyDrive/diffusion_checkpoints/best_model/weights",
    device="cuda",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-base")
print("Model loaded ✓")

In [ ]:
test_text = """Российские учёные из Института ядерной физики СО РАН разработали
новый метод диагностики материалов с помощью синхротронного излучения.
Метод позволяет исследовать внутреннюю структуру объектов без их разрушения.
Технология может применяться в медицине, промышленности и археологии.
Результаты исследования опубликованы в журнале Nature Materials."""

inputs = tokenizer(
    test_text, max_length=512, padding="max_length",
    truncation=True, return_tensors="pt",
).to(device)

with torch.no_grad():
    generated_ids, confidence = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128, num_inference_steps=10,
    )

summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Source:\n{test_text}\n")
print(f"Summary:\n{summary}\n")
print(f"Confidence: {confidence[0].mean():.4f}")

## 5. Download Best Weights

Weights are already on Google Drive at:

`/content/drive/MyDrive/diffusion_checkpoints/best_model/weights/`

Files: `model.pt`, `config.pt`, `tokenizer.json`

In [ ]:
# Show best model files on Google Drive
from pathlib import Path

weights_dir = Path("/content/drive/MyDrive/diffusion_checkpoints/best_model/weights")
if weights_dir.exists():
    print("Best model weights (on Google Drive):")
    total = 0
    for f in sorted(weights_dir.rglob("*")):
        if f.is_file():
            mb = f.stat().st_size / 1024 / 1024
            total += mb
            print(f"  {f.name}: {mb:.1f} MB")
    print(f"\n  Total: {total:.1f} MB")
    print(f"\n  Path: {weights_dir}")
else:
    print("Best model not saved yet.")